In [ ]:
import random
import collections
from pathlib import Path
import csv
from g2p_en import G2p

## loading g2p

g2p=G2p()
TEXT_FILE = 'text_360h'

lex_dict = {}
with open(TEXT_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, text = parts
        words = text.strip().split()
        for word in words: 
            if lex_dict.get(word) is None: 
                ph_str=g2p(word)
                joined_ph_str = " ".join(ph_str).replace('0','').replace('1','').replace('2','')
                lex_dict[word] = joined_ph_str

TEXT_FILE = 'text_100h'

with open(TEXT_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, text = parts
        words = text.strip().split()
        for word in words: 
            if lex_dict.get(word) is None: 
                ph_str=g2p(word)
                joined_ph_str = " ".join(ph_str).replace('0','').replace('1','').replace('2','')
                lex_dict[word] = joined_ph_str

TEXT_FILE = 'test_clean_text'

with open(TEXT_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, text = parts
        words = text.strip().split()
        for word in words: 
            if lex_dict.get(word) is None: 
                ph_str=g2p(word)
                joined_ph_str = " ".join(ph_str).replace('0','').replace('1','').replace('2','')
                lex_dict[word] = joined_ph_str

               

LEXICON_FILE = 'lexicon_g2p.txt'

with open(LEXICON_FILE  , 'w') as f:
    for file_id, ph_str in lex_dict.items():
        f.write(f"{file_id} {ph_str}\n")

In [ ]:
import random
import collections
from pathlib import Path
import csv
import json
from g2p_en import G2p
g2p = G2p()
from tqdm import tqdm

# --- Config: Updated paths ---
LEXICON_FILE = 'lexicon_g2p.txt'
TEXT_FILE = 'text_360h'
WAV_SCP_FILE = 'wav.scp_360h'
CTM_FILE = 'librispeech_clean_train_360h_all_utt.csv'

OUT_DIR = Path('train_360h')  # output directory
OUT_DIR.mkdir(exist_ok=True)

MIN_LENGTH = 4
MAX_LENGTH = 12

# --- Step 1: Load Lexicon ---
def load_lexicon(path):
    lex = {}
    with open(path, encoding='utf8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            word = parts[0].upper()
            phonemes = parts[1:]
            lex[word] = phonemes
    return lex

lexicon = load_lexicon(LEXICON_FILE)

# --- Step 2: Parse text; build mappings keyword -> utterances, freq, length ---
keyword_to_uttids = collections.defaultdict(set)   # keyword -> set(utt_id)
keyword_freq = collections.Counter()               # keyword -> total count
keyword_length = {}

with open(TEXT_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, sent = parts
        words = sent.strip().split()
        for word in words:
            w = word.upper()
            if w not in lexicon:
                continue
            length = len(lexicon[w])
            if length < MIN_LENGTH or length > MAX_LENGTH:
                continue
            keyword_to_uttids[w].add(utt_id)
            keyword_freq[w] += 1
            keyword_length[w] = length

# --- Step 3: Organize keywords by length and split keywords into IV/OOV ---
length_to_keywords = collections.defaultdict(list)
for kw, length in keyword_length.items():
    length_to_keywords[length].append(kw)

# --- Load wav.scp mapping utt_id -> wav_path ---
wavscp_map = {}
with open(WAV_SCP_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, wav_path = parts
        wavscp_map[utt_id] = wav_path

# --- Initialize dicts to store IV and OOV keywords by length ---
iv_dict = {}
oov_dict = {}

# --- Step 4: For each length, split keywords and write WAV SCP + frequency CSV ---
summary_rows = []

for length in range(MIN_LENGTH, MAX_LENGTH + 1):
    keywords = length_to_keywords.get(length, [])
    if not keywords:
        print(f"No keywords found for phoneme length {length}; skipping.")
        continue


    # Sort keywords by frequency descending
    keywords_sorted = sorted(keywords, key=lambda k: keyword_freq[k], reverse=True)
    top_15_pct_count = max(1, int(len(keywords_sorted) * 0.15))

    if top_15_pct_count < 60:
        print(f"Warning: For length {length}, top 15% count is {top_15_pct_count}, which is less than 60. Adjusting OOV count accordingly.")
        print("Change the top percentage or ensure enough keywords for this length to have a meaningful split.")
        break

    top_15_percent_keywords = keywords_sorted[:top_15_pct_count]
    rest_keywords = keywords_sorted[top_15_pct_count:]
    random.shuffle(top_15_percent_keywords)  # Shuffle to randomize IV/OOV selection

    IV_keywords_30 = top_15_percent_keywords[:30]  # Top 30 keywords from top 15% as IV (or fewer if top 15% has less than 30)
    OOV_keywords_30 = top_15_percent_keywords[30:60]  # Next 30 keywords from top 15% as OOV (or fewer if top 15% has less than 60)

    IV_keywords = set(IV_keywords_30) | set(rest_keywords) | (set(top_15_percent_keywords) - set(OOV_keywords_30))  # IV keywords are top 30 from top 15% + all keywords from rest + remaining keywords from top 15% except those chosen as OOV
    OOV_keywords = set(OOV_keywords_30)  # OOV keywords are next 30 from top 15%
    

    # Store the sorted lists in dicts

    # Store the sorted lists in dicts
    iv_dict[str(length)] = sorted(list(IV_keywords), key=lambda k: keyword_freq[k], reverse=True)
    oov_dict[str(length)] = sorted(list(OOV_keywords), key=lambda k: keyword_freq[k], reverse=True)
    iv_dict_30[str(length)] = sorted(IV_keywords_30, key=lambda k: keyword_freq[k], reverse=True)

    # Collect utterances for IV and OOV
    IV_utts = set()
    for kw in IV_keywords:
        IV_utts.update(keyword_to_uttids[kw])
    OOV_utts = set()
    for kw in OOV_keywords:
        OOV_utts.update(keyword_to_uttids[kw])



# --- Step 6: Save iv_dict and oov_dict as JSON ---
iv_dict_path = OUT_DIR / 'iv_dict.json'
with open(iv_dict_path, 'w', encoding='utf8') as f_iv:
    json.dump(iv_dict, f_iv, ensure_ascii=False, indent=2)

oov_dict_path = OUT_DIR / 'oov_dict.json'
with open(oov_dict_path, 'w', encoding='utf8') as f_oov:
    json.dump(oov_dict, f_oov, ensure_ascii=False, indent=2)
iv_dict_30_path = OUT_DIR / 'iv_dict_30.json'
with open(iv_dict_30_path, 'w', encoding='utf8') as f_iv30:
    json.dump(iv_dict_30, f_iv30, ensure_ascii=False, indent=2)
    

print("Done! All splits, frequency info, and iv/oov dicts are saved in the 'train_splits/' directory.")


# Get overall IV and OOV keywords (across all lengths)
overall_IV_keywords = set()
overall_OOV_keywords = set()

for kws in iv_dict.values():
    overall_IV_keywords.update(kws)
for kws in oov_dict.values():
    overall_OOV_keywords.update(kws)

print(f"Total unique IV keywords: {len(overall_IV_keywords)}")
print(f"Total unique OOV keywords: {len(overall_OOV_keywords)}")

# print("IV keywords:", overall_IV_keywords)
# print("OOV keywords:", overall_OOV_keywords)

uttid_to_words = {}
with open(TEXT_FILE, encoding='utf8') as f_text:
    for line in f_text:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, sent = parts
        words = set(w.upper() for w in sent.strip().split())
        uttid_to_words[utt_id] = words

f = open(CTM_FILE).read().splitlines()
iv_instances = []
oov_instances = []


iv_csv_path = OUT_DIR / 'iv_keyword_instances_in_ctm.csv'
oov_csv_path = OUT_DIR / 'oov_keyword_instances_in_ctm.csv'

with open(iv_csv_path, 'w', newline='', encoding='utf8') as iv_csvfile, \
     open(oov_csv_path, 'w', newline='', encoding='utf8') as oov_csvfile:

    iv_writer = csv.writer(iv_csvfile)
    oov_writer = csv.writer(oov_csvfile)

    iv_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time'])
    oov_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time'])


    for i in tqdm(range(len(f))):
        file_path, start, end, word = f[i].split(',')
        utt_id = 'lbi-' + file_path.split('/')[-1][:-4]
        word = word.upper()
        if word == '<UNK>' or len(word) < 1:
            continue
        start_time = int(float(start) * 100)
        end_time = int(float(end) * 100)
        words_in_utt = uttid_to_words.get(utt_id, set())
        # Positive IV instance
        if word in overall_IV_keywords:
            #iv_instances.append((utt_id, word, start_time, end_time))
            iv_writer.writerow([utt_id, word, start_time, end_time])
            
            # Sample negative word from utterance
            #overall_IV_keywords.difference_update(words_in_utt)

            # Select a random keyword from the remaining set
            #neg_word = random.choice(list(overall_IV_keywords))

            # Restore the original set by adding back the removed keywords
            #overall_IV_keywords.update(words_in_utt)
            
            
            # Remove IV keywords and the current word
            negative_candidates = overall_IV_keywords - uttid_to_words[utt_id]# - {word}
            
            neg_word = random.choice(list(negative_candidates))
            #print(word, neg_word)
            #iv_instances.append((utt_id, neg_word, 0, 0))
            iv_writer.writerow([utt_id, neg_word, 0, 0])

        if word in overall_OOV_keywords:
            #oov_instances.append((utt_id, word, start_time, end_time))
            oov_writer.writerow([utt_id, word, start_time, end_time])


Done! All splits, frequency info, and iv/oov dicts are saved in the 'train_splits/' directory.
Total unique IV keywords: 54836
Total unique OOV keywords: 270


100%|██████████| 4288538/4288538 [42:34<00:00, 1678.55it/s] 


# Validation

In [ ]:
import json
OUT_DIR = Path('train_360h')  # output directory
iv_dict_path = OUT_DIR / 'iv_dict.json'
with open(iv_dict_path, 'r', encoding='utf8') as f_iv:
    iv_dict = json.load(f_iv)
    
oov_dict_path = OUT_DIR / 'oov_dict.json'
with open(oov_dict_path, 'r', encoding='utf8') as f_oov:
    oov_dict = json.load(f_oov)

train_IV_keywords = set()
train_OOV_keywords = set()

for kws in iv_dict.values():
    train_IV_keywords.update(kws)
for kws in oov_dict.values():
    train_OOV_keywords.update(kws)

TEXT_FILE = 'text'
uttid_to_words = {}
with open(TEXT_FILE, encoding='utf8') as f_text:
    for line in f_text:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, sent = parts
        words = set(w.upper() for w in sent.strip().split())
        uttid_to_words[utt_id] = words


OUT_DIR = Path('test_clean')  # output directory
OUT_DIR.mkdir(exist_ok=True)
CTM_FILE = 'librispeech_clean_test_all_utt.csv'

f = open(CTM_FILE).read().splitlines()
with open(OUT_DIR / 'iv_keyword_instances_in_ctm.csv', 'w', newline='', encoding='utf8') as iv_csvfile, \
     open(OUT_DIR / 'oov_keyword_instances_in_ctm.csv', 'w', newline='', encoding='utf8') as oov_csvfile, \
     open(OUT_DIR / 'iv_or_oov_keyword_instances_in_ctm.csv', 'w', newline='', encoding='utf8') as overall_csvfile:

    iv_writer = csv.writer(iv_csvfile)
    oov_writer = csv.writer(oov_csvfile)
    overall_writer = csv.writer(overall_csvfile)

    iv_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time'])
    oov_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time'])
    overall_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time'])

    for i in tqdm(range(len(f))):
        file_path, start, end, word = f[i].split(',')
        utt_id = 'lbi-' + file_path.split('/')[-1][:-4]
        if len(word) >= 1:
            word = word.upper()
            if word == '<UNK>':
                continue
            start_time = int(float(start) * 100)
            end_time = int(float(end) * 100)

            if word in train_IV_keywords or word in train_OOV_keywords:
                # Sample negative word from utterance
                negative_candidates = train_IV_keywords - uttid_to_words[utt_id]# - {word}
                neg_word = random.choice(list(negative_candidates))
                if word in train_IV_keywords:
                    iv_writer.writerow([utt_id, word, start_time, end_time])
                    iv_writer.writerow([utt_id, neg_word, 0, 0])
                    
                else:
                    oov_writer.writerow([utt_id, word, start_time, end_time])
                
                overall_writer.writerow([utt_id, word, start_time, end_time])
                overall_writer.writerow([utt_id, neg_word, 0, 0])
            else:         
                continue


100%|██████████| 64221/64221 [00:41<00:00, 1544.18it/s]


# IV and OOV evaluation trails preparation

In [ ]:
import json
import random
import csv
from pathlib import Path


# --- Load IV/OOV dicts ---
OUT_DIR = Path('train_360h')

iv_dict_path = OUT_DIR / 'iv_dict_30.json'
with open(iv_dict_path, 'r', encoding='utf8') as f_iv:
    iv_dict = json.load(f_iv)
    
oov_dict_path = OUT_DIR / 'oov_dict.json'
with open(oov_dict_path, 'r', encoding='utf8') as f_oov:
    oov_dict = json.load(f_oov)
    
import random
import collections
from pathlib import Path
import csv
import json
from g2p_en import G2p
g2p = G2p()
from tqdm import tqdm

# --- Config: Updated paths ---
LEXICON_FILE = 'lexicon_g2p.txt'
TEXT_FILE = 'text_100h'
WAV_SCP_FILE = 'wav.scp_100h'
CTM_FILE = 'librispeech_clean_train_100h_all_utt.csv'
SPLIT_RATIO = 0.7  # train/test split

MIN_LENGTH = 4
MAX_LENGTH = 12

# --- Step 1: Load Lexicon ---
def load_lexicon(path):
    lex = {}
    with open(path, encoding='utf8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            word = parts[0].upper()
            phonemes = parts[1:]
            lex[word] = phonemes
    return lex

lexicon = load_lexicon(LEXICON_FILE)

# --- Step 2: Parse text; build mappings keyword -> utterances, freq, length ---
keyword_to_uttids = collections.defaultdict(set)   # keyword -> set(utt_id)
keyword_freq = collections.Counter()               # keyword -> total count
keyword_length = {}

with open(TEXT_FILE, encoding='utf8') as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) != 2:
            continue
        utt_id, sent = parts
        words = sent.strip().split()
        for word in words:
            w = word.upper()
            if w not in lexicon:
                continue
            length = len(lexicon[w])
            if length < MIN_LENGTH or length > MAX_LENGTH:
                continue
            keyword_to_uttids[w].add(utt_id)
            keyword_freq[w] += 1
            keyword_length[w] = length

# --- Parse CTM file ---
CTM_FILE = 'librispeech_clean_train_100h_all_utt.csv'
ctm_lines = open(CTM_FILE).read().splitlines()

# Build: keyword -> list of (utt_id, start_time, end_time)
ctm_dict = {}
all_utt_ids = set()
for line in ctm_lines:
    file_path, start, end, word = line.split(',')
    utt_id = 'lbi-' + file_path.split('/')[-1][:-4]
    word = word.upper()
    if word == '<UNK>' or not word:
        continue
    start_time = int(float(start) * 100)
    end_time = int(float(end) * 100)
    all_utt_ids.add(utt_id)
    if word not in ctm_dict:
        ctm_dict[word] = []
    ctm_dict[word].append((utt_id, start_time, end_time))

with open('train_100h/iv_trials_final.csv', 'w', newline='', encoding='utf8') as iv_f, \
     open('train_100h/oov_trials_final.csv', 'w', newline='', encoding='utf8') as oov_f:
    iv_writer = csv.writer(iv_f)
    oov_writer = csv.writer(oov_f)
    iv_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time', 'label','length'])
    oov_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time', 'label','length'])

    length_to_keywords = collections.defaultdict(list)
    
    # For each length, only use top 30 IV and OOV keywords
    for length in iv_dict:

        iv_keywords = iv_dict[length]
        
        keywords_sorted = sorted(iv_keywords, key=lambda k: keyword_freq[k], reverse=True)
        top_20_pct_count = max(1, int(len(keywords_sorted) * 0.15))

        top_20_percent_keywords = keywords_sorted[:top_20_pct_count]

        # Randomly pick 30 keywords from top 20% as OOV (or fewer if top 20% has less than 30)
        iv_30_count = min(30, len(top_20_percent_keywords))
        iv_keywords_30 = set(random.sample(top_20_percent_keywords, iv_30_count))
        
        #print()

        for kw in iv_keywords_30:
            length_to_keywords[length].append(kw)
            pos_list = ctm_dict.get(kw, [])
            pos_utt_ids = set([utt_id for utt_id, _, _ in pos_list])
            # Write positives
            for utt_id, start_time, end_time in pos_list:
                iv_writer.writerow([utt_id, kw, start_time, end_time, 1, length])
            # Write negatives: sample same number of utt_ids not containing the keyword
            neg_candidates = list(all_utt_ids - pos_utt_ids)
            if len(neg_candidates) >= len(pos_list):
                neg_utt_ids = random.sample(neg_candidates, len(pos_list))
            else:
                neg_utt_ids = neg_candidates
            for utt_id in neg_utt_ids:
                iv_writer.writerow([utt_id, kw, 0, 0, 0,length])

    iv_dict_path1 = OUT_DIR / 'iv_dict_30_keyword.json'
    with open(iv_dict_path1, 'w', encoding='utf8') as f_iv:
        json.dump(length_to_keywords, f_iv, indent=2)
        
    for length in oov_dict:
        
        oov_keywords = oov_dict[length][:30]
        for kw in oov_keywords:
            pos_list = ctm_dict.get(kw, [])
            pos_utt_ids = set([utt_id for utt_id, _, _ in pos_list])
            for utt_id, start_time, end_time in pos_list:
                oov_writer.writerow([utt_id, kw, start_time, end_time, 1,length])
            neg_candidates = list(all_utt_ids - pos_utt_ids)
            if len(neg_candidates) >= len(pos_list):
                neg_utt_ids = random.sample(neg_candidates, len(pos_list))
            else:
                neg_utt_ids = neg_candidates
            for utt_id in neg_utt_ids:
                oov_writer.writerow([utt_id, kw, 0, 0, 0,length])

# Train -other

In [ ]:
import json
import random
import csv
from pathlib import Path


# --- Load IV/OOV dicts ---
OUT_DIR = Path('train_360h')
iv_dict_path = OUT_DIR / 'iv_dict_30_keyword.json'
oov_dict_path = OUT_DIR / 'oov_dict.json'
with open(iv_dict_path, 'r', encoding='utf8') as f_iv:
    iv_dict = json.load(f_iv)
with open(oov_dict_path, 'r', encoding='utf8') as f_oov:
    oov_dict = json.load(f_oov)

import random
import collections
from pathlib import Path
import csv
import json
from g2p_en import G2p
g2p = G2p()
from tqdm import tqdm

# --- Config: Updated paths ---
LEXICON_FILE = 'lexicon_g2p.txt'
TEXT_FILE = 'text_100h'
#WAV_SCP_FILE = 'wav.scp_100h'
CTM_FILE = 'librispeech_other_train_500h_all_utt.csv'
SPLIT_RATIO = 0.7  # train/test split

MIN_LENGTH = 4
MAX_LENGTH = 12

# --- Step 1: Load Lexicon ---
def load_lexicon(path):
    lex = {}
    with open(path, encoding='utf8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2:
                continue
            word = parts[0].upper()
            phonemes = parts[1:]
            lex[word] = phonemes
    return lex

lexicon = load_lexicon(LEXICON_FILE)

# --- Step 2: Parse text; build mappings keyword -> utterances, freq, length ---
keyword_to_uttids = collections.defaultdict(set)   # keyword -> set(utt_id)
keyword_freq = collections.Counter()               # keyword -> total count
keyword_length = {}

# with open(TEXT_FILE, encoding='utf8') as f:
#     for line in f:
#         parts = line.strip().split(None, 1)
#         if len(parts) != 2:
#             continue
#         utt_id, sent = parts
#         words = sent.strip().split()
#         for word in words:
#             w = word.upper()
#             if w not in lexicon:
#                 continue
#             length = len(lexicon[w])
#             if length < MIN_LENGTH or length > MAX_LENGTH:
#                 continue
#             keyword_to_uttids[w].add(utt_id)
#             keyword_freq[w] += 1
#             keyword_length[w] = length

# --- Parse CTM file ---
ctm_lines = open(CTM_FILE).read().splitlines()

# Build: keyword -> list of (utt_id, start_time, end_time)
ctm_dict = {}
all_utt_ids = set()
for line in ctm_lines:
    file_path, start, end, word = line.split(',')
    utt_id = 'lbi-' + file_path.split('/')[-1][:-4]
    word = word.upper()
    if word == '<UNK>' or not word:
        continue
    start_time = int(float(start) * 100)
    end_time = int(float(end) * 100)
    all_utt_ids.add(utt_id)
    if word not in ctm_dict:
        ctm_dict[word] = []
    ctm_dict[word].append((utt_id, start_time, end_time))

with open('train_500h/iv_trials_final.csv', 'w', newline='', encoding='utf8') as iv_f, \
     open('train_500h/oov_trials_final.csv', 'w', newline='', encoding='utf8') as oov_f:
    iv_writer = csv.writer(iv_f)
    oov_writer = csv.writer(oov_f)
    iv_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time', 'label','length'])
    oov_writer.writerow(['utt_id', 'keyword', 'start_time', 'end_time', 'label','length'])

    length_to_keywords = collections.defaultdict(list)
    
    # For each length, only use top 30 IV and OOV keywords
    for length in iv_dict:

        iv_keywords_30 = iv_dict[length]
        
        #keywords_sorted = sorted(iv_keywords, key=lambda k: keyword_freq[k], reverse=True)
        #top_20_pct_count = max(1, int(len(keywords_sorted) * 0.15))

        #top_20_percent_keywords = keywords_sorted[:top_20_pct_count]

        # Randomly pick 30 keywords from top 20% as OOV (or fewer if top 20% has less than 30)
        #iv_30_count = min(30, len(top_20_percent_keywords))
        #iv_keywords_30 = set(random.sample(top_20_percent_keywords, iv_30_count))
        
        #print()

        for kw in iv_keywords_30:
            #length_to_keywords[length].append(kw)
            pos_list = ctm_dict.get(kw, [])
            pos_utt_ids = set([utt_id for utt_id, _, _ in pos_list])
            # Write positives
            for utt_id, start_time, end_time in pos_list:
                iv_writer.writerow([utt_id, kw, start_time, end_time, 1, length])
            # Write negatives: sample same number of utt_ids not containing the keyword
            neg_candidates = list(all_utt_ids - pos_utt_ids)
            if len(neg_candidates) >= len(pos_list):
                neg_utt_ids = random.sample(neg_candidates, len(pos_list))
            else:
                neg_utt_ids = neg_candidates
            for utt_id in neg_utt_ids:
                iv_writer.writerow([utt_id, kw, 0, 0, 0,length])

    #iv_dict_path1 = OUT_DIR / 'iv_dict_30_keyword.json'
    #with open(iv_dict_path1, 'w', encoding='utf8') as f_iv:
    #    json.dump(length_to_keywords, f_iv, indent=2)
        
    for length in oov_dict:
        
        oov_keywords = oov_dict[length][:30]
        for kw in oov_keywords:
            pos_list = ctm_dict.get(kw, [])
            pos_utt_ids = set([utt_id for utt_id, _, _ in pos_list])
            for utt_id, start_time, end_time in pos_list:
                oov_writer.writerow([utt_id, kw, start_time, end_time, 1,length])
            neg_candidates = list(all_utt_ids - pos_utt_ids)
            if len(neg_candidates) >= len(pos_list):
                neg_utt_ids = random.sample(neg_candidates, len(pos_list))
            else:
                neg_utt_ids = neg_candidates
            for utt_id in neg_utt_ids:
                oov_writer.writerow([utt_id, kw, 0, 0, 0,length])